# Inc. 5000 Analysis

## Notebook 02 - Data Validation

### Objectives

This notebook validates the quality of the cleaned dataset before performing exploratory data analysis.

### Validation Checklist

- Dataset dimensions
- Missing values
- Duplicate records
- Data types
- Unique values
- Invalid values
- Numerical range checks
- Outlier detection
- Categorical consistency
- Final data quality report

---

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
# Path to dataset

file_path = r"C:\Users\dell\Desktop\Project\data\cleaned\inc5000_cleaned.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## Dataset Shape

In [3]:
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Rows    : 5,000
Columns : 21


## Missing Value Analysis

In [4]:
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Percentage": (
        df.isna().sum() / len(df) * 100
    ).round(2)
})

missing.sort_values(
    "Missing Values",
    ascending=False
)

,Missing Values,Percentage
metro,58,1.16
rank,0,0.00
employee_count,0,0.00
company,0,0.00
id,0,0.00
state_name,0,0.00
state_code,0,0.00
city,0,0.00
growth_percentage,0,0.00
revenue_usd,0,0.00


In [5]:
total_missing = df.isna().sum().sum()

print(f"Total Missing Values : {total_missing:,}")

Total Missing Values : 58


## Duplicate Records

In [6]:
duplicates = df.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

Duplicate Rows : 0


In [7]:
if duplicates > 0:
    display(df[df.duplicated()])
else:
    print("No duplicate records found.")

No duplicate records found.


## Data Types

In [8]:
dtype_df = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.values
})

dtype_df

,Column,Data Type
0,id,int64
1,rank,int64
2,employee_count,int64
3,company,str
4,state_name,str
5,state_code,str
6,city,str
7,metro,str
8,growth_percentage,float64
9,revenue_usd,int64


## Unique Values

In [9]:
unique_values = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": df.nunique()
})

unique_values.sort_values(
    "Unique Values",
    ascending=False
)

,Column,Unique Values
id,id,5000
rank,rank,5000
company,company,5000
growth_percentage,growth_percentage,4998
revenue_usd,revenue_usd,4995
Revenue_in_million,Revenue_in_million,4995
revenue_per_employee,revenue_per_employee,4986
city,city,1352
employee_count,employee_count,657
metro,metro,325


## Numerical Summary

In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,"5,000.00","20,036.58","8,041.04",4.00,"19,575.25","23,291.50","25,370.25","26,620.00"
rank,"5,000.00","2,500.50","1,443.52",1.00,"1,250.75","2,500.50","3,750.25","5,000.00"
employee_count,"5,000.00",208.97,"1,074.60",0.00,24.00,50.00,125.00,"34,219.00"
growth_percentage,"5,000.00",516.44,"2,786.06",42.45,84.21,151.72,347.65,"158,956.91"
revenue_usd,"5,000.00","43,058,182.36","181,855,882.83","1,953,000.00","4,876,790.75","10,722,077.00","26,952,131.25","5,528,202,691.00"
years_on_list,"5,000.00",2.74,2.00,1.00,1.00,2.00,4.00,12.00
Revenue_in_million,"5,000.00","43,058,182.36","181,855,882.83","1,953,000.00","4,876,790.75","10,722,077.00","26,952,131.25","5,528,202,691.00"


## Negative Value Check

In [13]:
numeric_columns = df.select_dtypes(include="number").columns

negative_counts = {}

for col in numeric_columns:
    negative_counts[col] = (df[col] < 0).sum()

negative_df = pd.DataFrame({
    "Column": negative_counts.keys(),
    "Negative Values": negative_counts.values()
})

negative_df

,Column,Negative Values
0,id,0
1,rank,0
2,employee_count,0
3,growth_percentage,0
4,revenue_usd,0
5,years_on_list,0
6,Revenue_in_million,0


## Zero Value Check

In [14]:
zero_counts = {}

for col in numeric_columns:
    zero_counts[col] = (df[col] == 0).sum()

pd.DataFrame({
    "Column": zero_counts.keys(),
    "Zero Values": zero_counts.values()
})

,Column,Zero Values
0,id,0
1,rank,0
2,employee_count,13
3,growth_percentage,0
4,revenue_usd,0
5,years_on_list,0
6,Revenue_in_million,0


## Outlier Detection (IQR Method)

In [15]:
outlier_summary = []

for col in numeric_columns:

    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ].shape[0]

    outlier_summary.append({
        "Column": col,
        "Outliers": outliers
    })

pd.DataFrame(outlier_summary).sort_values(
    "Outliers",
    ascending=False
)

,Column,Outliers
0,id,834
3,growth_percentage,627
4,revenue_usd,613
6,Revenue_in_million,613
2,employee_count,596
5,years_on_list,37
1,rank,0


## Blank String Validation

In [16]:
object_columns = df.select_dtypes(include="object").columns

blank_summary = []

for col in object_columns:

    blanks = (
        df[col]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    blank_summary.append({
        "Column": col,
        "Blank Values": blanks
    })

pd.DataFrame(blank_summary)

C:\Users\dell\AppData\Local\Temp\ipykernel_20124\1514776582.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include="object").columns


,Column,Blank Values
0,company,0
1,state_name,0
2,state_code,0
3,city,0
4,metro,0
5,industry,0
6,revenue_per_employee,0
7,company_size,0
8,employee_size_band,0
9,revenue_category,0


## Leading / Trailing Spaces

In [17]:
space_summary = []

for col in object_columns:

    spaces = (
        df[col]
        .astype(str)
        .str.match(r"^\s|\s$")
        .sum()
    )

    space_summary.append({
        "Column": col,
        "Leading/Trailing Spaces": spaces
    })

pd.DataFrame(space_summary)

,Column,Leading/Trailing Spaces
0,company,0
1,state_name,0
2,state_code,0
3,city,0
4,metro,0
5,industry,0
6,revenue_per_employee,0
7,company_size,0
8,employee_size_band,0
9,revenue_category,0


## Company Name Validation

In [25]:
if "company" in df.columns:

    duplicate_companies = (
        df["company"]
        .duplicated()
        .sum()
    )

    print("Duplicate Company Names:", duplicate_companies)

else:

    print("Company column not found.")

Duplicate Company Names: 0


## State Validation

In [24]:
if "state_name" in df.columns:

    print(df["state_name"].value_counts())

else:

    print("State column not found.")

state_name
California              694
Texas                   404
New York                335
Florida                 303
Virginia                284
Illinois                238
Georgia                 209
Pennsylvania            191
Massachusetts           174
Ohio                    171
New Jersey              164
North Carolina          146
Michigan                130
Colorado                122
Maryland                121
Washington              107
Arizona                 104
Minnesota                94
Utah                     86
Tennessee                80
Indiana                  73
Missouri                 72
Wisconsin                70
Oregon                   61
Alabama                  56
South Carolina           55
Connecticut              47
District of Columbia     44
Louisiana                41
Kansas                   36
Kentucky                 31
Nevada                   31
Iowa                     30
Oklahoma                 30
Nebraska                 28
New Hamps

## Revenue Validation

In [22]:
if "revenue_usd" in df.columns:

    print(df["revenue_usd"].describe())

else:

    print("revenue_usd column not found.")

count           5,000.00
mean       43,058,182.36
std       181,855,882.83
min         1,953,000.00
25%         4,876,790.75
50%        10,722,077.00
75%        26,952,131.25
max     5,528,202,691.00
Name: revenue_usd, dtype: float64


In [21]:
## Employee Validation

In [26]:
if "employee_count" in df.columns:

    print(df["employee_count"].describe())

else:

    print("Employees column not found.")

count    5,000.00
mean       208.97
std      1,074.60
min          0.00
25%         24.00
50%         50.00
75%        125.00
max     34,219.00
Name: employee_count, dtype: float64


## Data Quality Summary

In [28]:
quality_report = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows",
        "Numeric Columns",
        "Categorical Columns"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        df.isna().sum().sum(),
        df.duplicated().sum(),
        len(numeric_columns),
        len(object_columns)
    ]
})

quality_report

,Metric,Value
0,Rows,5000
1,Columns,21
2,Missing Values,58
3,Duplicate Rows,0
4,Numeric Columns,7
5,Categorical Columns,14


In [29]:
issues = []

if df.isna().sum().sum() > 0:
    issues.append("Missing values detected.")

if df.duplicated().sum() > 0:
    issues.append("Duplicate rows detected.")

if not issues:
    issues.append("No major data quality issues detected.")

print("Validation Result")
print("-" * 40)

for issue in issues:
    print(f"• {issue}")

Validation Result
----------------------------------------
• Missing values detected.


# Conclusion

The dataset has been validated for:

- Dataset dimensions
- Missing values
- Duplicate records
- Data types
- Unique values
- Numerical consistency
- Zero values
- Negative values
- Outlier detection
- Blank text values
- Leading/trailing spaces
- Company validation
- State validation
- Revenue validation
- Employee validation

The dataset is now ready for exploratory data analysis.

**Next Notebook:** `03_eda.ipynb`